# Colab — duże enkodery (HerBERT-large + XLM-R-large), pełny fine-tune

Dwa enkodery na TwitterEmo, recepta jak lokalnie (pos_weight + early stopping + best-epoch).
T4 16 GB wystarczy, bez LoRA i kwantyzacji.

Runtime → Change runtime type → GPU (T4). Komórka 1 poprosi o `kaggle.json`, żeby pobrać dane.

In [ ]:
!pip install -q -U "transformers>=4.44" "datasets>=2.20" accelerate kaggle 2>/dev/null
import os, glob
# --- WKLEJ swój klucz Kaggle (Settings → API → Create New API Token).
#     Bez okienka uploadu — kaggle CLI czyta zmienne środowiskowe.
#     UWAGA: nie udostępniaj notebooka z kluczem w środku! ---
os.environ["KAGGLE_USERNAME"] = "miczimici"
os.environ["KAGGLE_KEY"]      = "WKLEJ_TUTAJ_KLUCZ"      # <-- tu Twój klucz
if not glob.glob("/content/**/twitteremo_train.csv", recursive=True):
    !kaggle datasets download -d miczimici/pl-emotion-processed -p /content/data --unzip
print("CSV:", glob.glob("/content/**/twitteremo_*.csv", recursive=True)[:3])

In [ ]:
import numpy as np, pandas as pd, torch, torch.nn.functional as F, glob
from scipy.special import expit
from sklearn.metrics import (f1_score, hamming_loss, jaccard_score, accuracy_score, precision_score, recall_score)
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding, EarlyStoppingCallback)
import warnings; warnings.filterwarnings("ignore")
RANDOM_STATE=42; torch.manual_seed(RANDOM_STATE); np.random.seed(RANDOM_STATE)
EMOTIONS=["radość","smutek","zaufanie","wstręt","strach","gniew","przeczuwanie","zdziwienie"]
def f(n): return glob.glob(f"/content/**/{n}", recursive=True)[0]
tw_train=pd.read_csv(f("twitteremo_train.csv")); tw_val=pd.read_csv(f("twitteremo_val.csv")); tw_test=pd.read_csv(f("twitteremo_test.csv"))
for d in (tw_train,tw_val,tw_test): d["tekst"]=d["tekst"].fillna("")
y_val,y_test=tw_val[EMOTIONS].values,tw_test[EMOTIONS].values
pos=tw_train[EMOTIONS].values.sum(0); neg=len(tw_train)-pos
POS_WEIGHT=torch.tensor(np.clip(neg/np.maximum(pos,1),1.0,10.0),dtype=torch.float32)

# --- zamontuj Drive TERAZ (autoryzacja gdy jesteś przy kompie) -> późniejsze zapisy ciche ---
import os
try:
    from google.colab import drive; drive.mount("/content/drive")
    DRIVE_DIR="/content/drive/MyDrive/master-thesis"; os.makedirs(DRIVE_DIR, exist_ok=True)
    print("Drive OK ->", DRIVE_DIR)
except Exception as e:
    DRIVE_DIR=None; print("Drive niedostępny (zapis tylko lokalnie):", e)

print("train",len(tw_train),"| gpu",torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

In [ ]:
def evaluate(yt,yp):
    return {"f1_macro":f1_score(yt,yp,average="macro",zero_division=0),"f1_micro":f1_score(yt,yp,average="micro",zero_division=0),
            "jaccard_macro":jaccard_score(yt,yp,average="macro",zero_division=0),"hamming_loss":hamming_loss(yt,yp)}
def opt_thr(yt,yp):
    thr=np.full(len(EMOTIONS),0.5)
    for i in range(len(EMOTIONS)):
        bf,bt=0.0,0.5
        for t in np.arange(0.05,0.95,0.01):
            v=f1_score(yt[:,i],(yp[:,i]>=t).astype(int),zero_division=0)
            if v>bf: bf,bt=v,t
        thr[i]=bt
    return thr
def f1_ci(yt,yp,n=1000,seed=RANDOM_STATE):
    rng=np.random.default_rng(seed); m=len(yt); base=f1_score(yt,yp,average="macro",zero_division=0)
    b=[f1_score(yt[i],yp[i],average="macro",zero_division=0) for i in (rng.integers(0,m,m) for _ in range(n))]
    lo,hi=np.percentile(b,[2.5,97.5]); return base,lo,hi
class WTrainer(Trainer):
    def __init__(self,*a,pos_weight=None,**k): super().__init__(*a,**k); self.pw=pos_weight
    def compute_loss(self,model,inputs,return_outputs=False,**kw):
        lab=inputs.pop("labels"); out=model(**inputs)
        loss=F.binary_cross_entropy_with_logits(out.logits.float(),lab.float(),pos_weight=self.pw.to(out.logits.device))
        return (loss,out) if return_outputs else loss
def train_one(model_name,batch,epochs=3,lr=2e-5):
    tok=AutoTokenizer.from_pretrained(model_name)
    def to_ds(df):
        d=Dataset.from_dict({"text":df["tekst"].tolist(),"labels":df[EMOTIONS].values.astype("float32").tolist()})
        return d.map(lambda b: tok(b["text"],truncation=True,max_length=128),batched=True,remove_columns=["text"])
    dtr,dva,dte=to_ds(tw_train),to_ds(tw_val),to_ds(tw_test)
    model=AutoModelForSequenceClassification.from_pretrained(model_name,num_labels=len(EMOTIONS),problem_type="multi_label_classification")
    args=TrainingArguments(output_dir=f"/content/ck_{model_name.split('/')[-1]}",eval_strategy="epoch",save_strategy="epoch",
        save_total_limit=1,load_best_model_at_end=True,metric_for_best_model="f1_macro",greater_is_better=True,
        per_device_train_batch_size=batch,per_device_eval_batch_size=32,gradient_accumulation_steps=2,gradient_checkpointing=True,
        num_train_epochs=epochs,learning_rate=lr,warmup_ratio=0.1,max_grad_norm=1.0,weight_decay=0.01,fp16=True,logging_steps=100,report_to="none",seed=RANDOM_STATE)
    cm=lambda p:{"f1_macro":f1_score(p.label_ids.astype(int),(expit(p.predictions)>=0.5).astype(int),average="macro",zero_division=0)}
    tr=WTrainer(model=model,args=args,train_dataset=dtr,eval_dataset=dva,data_collator=DataCollatorWithPadding(tok),
        compute_metrics=cm,pos_weight=POS_WEIGHT,callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
    tr.train()
    pv=expit(tr.predict(dva).predictions); pt=expit(tr.predict(dte).predictions)
    thr=opt_thr(y_val,pv); pred=(pt>=thr).astype(int); m=evaluate(y_test,pred); base,lo,hi=f1_ci(y_test,pred)
    m.update({"model":model_name.split("/")[-1]+"-full","ci_low":round(lo,3),"ci_high":round(hi,3)})
    del tr,model; torch.cuda.empty_cache(); print(f"  {m['model']}: F1-Macro={base:.3f} [{lo:.3f},{hi:.3f}]"); return m

In [ ]:
def save_results(rows):
    df = pd.DataFrame(rows)
    if DRIVE_DIR:
        df.to_csv(f"{DRIVE_DIR}/colab_large_results.csv", index=False)   # trwałe (Drive)
    df.to_csv("/content/colab_large_results.csv", index=False)
    return df

# Re-run XLM-R-large z NIŻSZYM LR (naprawa niestabilności). HerBERT-large gotowy (0,590) -> zakomentowany.
rows = []
for model_name, batch, lr in [
    # ("allegro/herbert-large-cased", 16, 2e-5),     # gotowe (0,590) — odkomentuj, by powtórzyć
    ("FacebookAI/xlm-roberta-large", 8, 1e-5),         # niższy LR + grad clipping -> stabilny fine-tune
]:
    rows.append(train_one(model_name, batch, lr=lr))
    save_results(rows)
    print(">>> ZAPISANO na Drive po:", rows[-1]["model"])

res = save_results(rows)
try:
    from google.colab import files; files.download("/content/colab_large_results.csv")
except Exception:
    pass
res[["model","f1_macro","ci_low","ci_high","f1_micro","jaccard_macro"]]

## Wynik

`colab_large_results.csv` (HerBERT-large-full + XLM-R-large-full, F1-Macro + 95% CI) —
zapisywany na Google Drive, lokalnie i przez auto-pobranie.